## Objectif de cette analyse

L’objectif de cette analyse est de comprendre la répartition géographique des niveaux de vie et d’identifier les disparités socio-économiques entre territoires. À partir du revenu médian, du taux de pauvreté et des indices d’inégalités on voudrait être capable de :
- Mesurer les contrastes régionaux entre zones aisées et zones défavorisées

- Analyser la structure du revenu (part des salaires, retraites, aides sociales) selon les départements

- Observer le lien entre pauvreté et inégalités

In [4]:
# install necessary packages
install.packages("readxl")
install.packages("dplyr")
install.packages("tidyr")
install.packages("ggplot2")
install.packages("maps")
install.packages("mapproj")
install.packages("FactoMineR")
install.packages("factoextra")
install.packages("gplots")
install.packages("corrplot")
install.packages("ellipse")
install.packages("car")
install.packages("lmtest")
install.packages("writexl")

# Load necessary packages
library(readxl)
library(dplyr)
library(tidyr)
library(ggplot2)
library(maps)
library(mapproj)
library(FactoMineR)
library(factoextra)
library(gplots)
library(corrplot)
library(ellipse)
library(car)
library(lmtest)
library(writexl)



# Plotting options
options(repr.plot.width = 14, repr.plot.height = 8)

also installing the dependency ‘cpp11’





The downloaded binary packages are in
	/var/folders/6t/l3cgh06n1rj2wpckspwwyc3m0000gn/T//RtmpiB3t6I/downloaded_packages

The downloaded binary packages are in
	/var/folders/6t/l3cgh06n1rj2wpckspwwyc3m0000gn/T//RtmpiB3t6I/downloaded_packages

The downloaded binary packages are in
	/var/folders/6t/l3cgh06n1rj2wpckspwwyc3m0000gn/T//RtmpiB3t6I/downloaded_packages

The downloaded binary packages are in
	/var/folders/6t/l3cgh06n1rj2wpckspwwyc3m0000gn/T//RtmpiB3t6I/downloaded_packages

The downloaded binary packages are in
	/var/folders/6t/l3cgh06n1rj2wpckspwwyc3m0000gn/T//RtmpiB3t6I/downloaded_packages

The downloaded binary packages are in
	/var/folders/6t/l3cgh06n1rj2wpckspwwyc3m0000gn/T//RtmpiB3t6I/downloaded_packages

The downloaded binary packages are in
	/var/folders/6t/l3cgh06n1rj2wpckspwwyc3m0000gn/T//RtmpiB3t6I/downloaded_packages

The downloaded binary packages are in
	/var/folders/6t/l3cgh06n1rj2wpckspwwyc3m0000gn/T//RtmpiB3t6I/downloaded_packages

The downloaded binary packages 

## Présentation du jeu de données

Le jeu de données que nous analisions provient de la base FILOSOFI 2018 (Fichier Localisé Social et Fiscal) publiée par l’INSEE. 
Cette base combine des données fiscales anonymisées issues de la Direction Générale des Finances Publiques (DGFiP) avec des données démographiques de l’INSEE.

In [6]:
# Read the data from the Excel file
data <- read_excel("datasets.xlsx", sheet = 1)

# Convert the data to a data frame
data <- as.data.frame(data)

dim(data)
names(data)

[1] 12395    30

[1] "IRIS"                                                 
 [2] "Libellé de l'IRIS"                                    
 [3] "Commune ou ARM"                                       
 [4] "Libellé commune ou ARM"                               
 [5] "Département"                                          
 [6] "Libellé département"                                  
 [7] "Part des ménages fiscaux imposés (%)"                 
 [8] "Taux de bas revenus déclarés au seuil de 60 % (%)"    
 [9] "1er quartile (€)"                                     
[10] "Médiane (€)"                                          
[11] "3e quartile (€)"                                      
[12] "Écart inter-quartile rapporté à la médiane"           
[13] "1er décile (€)"                                       
[14] "2e décile (€)"                                        
[15] "3e décile (€)"                                        
[16] "4e décile (€)"                                        
[17] "6e décile (€)"                                        
[18] "7e décile (€)"                                        
[19] "8e décile (€)"                                        
[20] "9e décile (€)"                                        
[21] "Rapport interdécile D9/D1"                            
[22] "S80/S20"                                              
[23] "Indice de Gini"                                       
[24] "Part des revenus d’activité (%)"                      
[25] "dont part des salaires et traitements (%)"            
[26] "dont part des indemnités de chômage (%)"              
[27] "dont part des revenus des activités non salariées (%)"
[28] "Part des pensions, retraites et rentes (%)"           
[29] "Part des autres revenus (%)"                          
[30] "Note de précaution (voir onglet seuils)"

On renomme les colonnes avec des noms courts. Ce sont les codes officiels de l'INSEE :

| Code | Signification |
|---|---|
| PIMP | Part des ménages fiscaux imposés (%) |
| TP60 | Taux de bas revenus au seuil de 60 % (%), notre indicateur de pauvreté |
| Q1, MED, Q3 | 1er quartile, médiane, 3e quartile du revenu par UC (€) |
| D1 … D9 | Déciles du revenu par UC (€) |
| EQ | Écart interquartile / médiane |
| RD | Rapport interdécile D9/D1 |
| S80S20 | Ratio S80/S20 |
| GINI | Indice de Gini |
| PACT | Part des revenus d'activité (%) = PTSA + PCHO + PBEN |
| PTSA | dont salaires et traitements (%) |
| PCHO | dont indemnités de chômage (%) |
| PBEN | dont revenus des activités non salariées (%) |
| PPEN | Part des pensions, retraites et rentes (%) |
| PAUT | Part des autres revenus, surtout du patrimoine (%) |
| NOTE | Note de précaution de l'INSEE |

In [9]:
# Rename the columns
rename_columns <- data
names(rename_columns) <- c("IRIS", "LIBIRIS", "COM", "LIBCOM", "DEP", "LIBDEP", "PIMP", "TP60", "Q1", "MED", "Q3", "EQ", "D1", "D2", "D3", "D4", "D6", "D7", "D8", "D9", "RD", "S80S20", "GINI", "PACT", "PTSA", "PCHO", "PBEN", "PPEN", "PAUT", "NOTE")

# Check the structure and first few rows of the renamed data frame
str(rename_columns)
head(rename_columns)

'data.frame':	12395 obs. of  30 variables:
 $ IRIS   : chr  "010040101" "010040102" "010040201" "010040202" ...
 $ LIBIRIS: chr  "Les Perouses-Triangle d'Activité" "Longeray-Gare" "Centre-Saint-Germain-Vareilles" "Tiret-Les Allymes" ...
 $ COM    : chr  "01004" "01004" "01004" "01004" ...
 $ LIBCOM : chr  "Ambérieu-en-Bugey" "Ambérieu-en-Bugey" "Ambérieu-en-Bugey" "Ambérieu-en-Bugey" ...
 $ DEP    : chr  "01" "01" "01" "01" ...
 $ LIBDEP : chr  "Ain" "Ain" "Ain" "Ain" ...
 $ PIMP   : num  45.4 39.3 43.7 61.1 41.9 46.7 48.6 39.2 36.7 54.5 ...
 $ TP60   : num  31 39.4 28.8 14.6 25.9 21.2 20.6 34.5 29.9 11.4 ...
 $ Q1     : num  11150 8890 11830 17180 12660 ...
 $ MED    : num  18540 16180 18870 24780 19590 ...
 $ Q3     : num  25900 23290 26560 33280 28730 ...
 $ EQ     : num  0.8 0.89 0.78 0.65 0.82 0.86 0.84 0.91 0.89 0.75 ...
 $ D1     : num  5500 3800 6770 10380 6620 ...
 $ D2     : num  9490 7690 9990 15570 10530 ...
 $ D3     : num  12830 10230 13210 19020 13900 ...
 $ D4     : num

,IRIS,LIBIRIS,COM,LIBCOM,DEP,LIBDEP,PIMP,TP60,Q1,MED,⋯,RD,S80S20,GINI,PACT,PTSA,PCHO,PBEN,PPEN,PAUT,NOTE
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,010040101,Les Perouses-Triangle d'Activité,01004,Ambérieu-en-Bugey,01,Ain,45.4,31.0,11150,18540,⋯,6.2,7.2,0.332,68.9,63.2,3.1,2.6,27.3,3.8,0
2,010040102,Longeray-Gare,01004,Ambérieu-en-Bugey,01,Ain,39.3,39.4,8890,16180,⋯,8.3,9.8,0.366,69.6,62.9,4.5,2.2,26.9,3.5,0
3,010040201,Centre-Saint-Germain-Vareilles,01004,Ambérieu-en-Bugey,01,Ain,43.7,28.8,11830,18870,⋯,5.2,7.4,0.347,67.7,60.6,3.1,4.0,28.3,4.0,0
4,010040202,Tiret-Les Allymes,01004,Ambérieu-en-Bugey,01,Ain,61.1,14.6,17180,24780,⋯,4.3,7.3,0.387,62.7,57.5,2.1,3.1,21.2,16.1,0
5,010330102,Centre Ville,01033,Valserhône,01,Ain,41.9,25.9,12660,19590,⋯,6.2,8.6,0.365,70.2,63.8,4.1,2.3,25.8,4.0,0
6,010330103,Lancrans-Coupy-Vanchy,01033,Valserhône,01,Ain,46.7,21.2,14700,22970,⋯,6.3,7.4,0.358,75.8,70.4,3.5,1.9,19.4,4.8,0


## Préparation des données

- On supprime les IRIS dont la note de précaution vaut 5 car ce sont les données jugées anormales par l'INSEE.
- On compte les valeurs manquantes, qui viennent du secret statistique.
- On ajoute la région, une variable qualitative.

In [12]:
# Remove rows with NOTE equal to 5
table(rename_columns$NOTE)
rename_columns <- rename_columns[rename_columns$NOTE != 5, ]

# Check the dimensions of the data frame after removing rows
dim(rename_columns)

# Missing values per variable
colSums(is.na(rename_columns))


    0     1 
12262   132 

[1] 12394    30

IRIS LIBIRIS     COM  LIBCOM     DEP  LIBDEP    PIMP    TP60      Q1     MED 
      0       0       0       0       0       0    1004    1796       0       0 
     Q3      EQ      D1      D2      D3      D4      D6      D7      D8      D9 
      0       0       0       0       0       0       0       0       0       0 
     RD  S80S20    GINI    PACT    PTSA    PCHO    PBEN    PPEN    PAUT    NOTE 
    654      39       0       0     440     440     440       0       0       0

In [14]:
# Region of each department
rename_columns$REGION <- NA
rename_columns$REGION[rename_columns$DEP %in% c("01","03","07","15","26","38","42","43","63","69","73","74")] <- "Auvergne-Rhone-Alpes"
rename_columns$REGION[rename_columns$DEP %in% c("21","25","39","58","70","71","89","90")]                     <- "Bourgogne-Franche-Comte"
rename_columns$REGION[rename_columns$DEP %in% c("22","29","35","56")]                                         <- "Bretagne"
rename_columns$REGION[rename_columns$DEP %in% c("18","28","36","37","41","45")]                               <- "Centre-Val de Loire"
rename_columns$REGION[rename_columns$DEP %in% c("2A","2B")]                                                   <- "Corse"
rename_columns$REGION[rename_columns$DEP %in% c("08","10","51","52","54","55","57","67","68","88")]           <- "Grand Est"
rename_columns$REGION[rename_columns$DEP %in% c("02","59","60","62","80")]                                    <- "Hauts-de-France"
rename_columns$REGION[rename_columns$DEP %in% c("75","77","78","91","92","93","94","95")]                     <- "Ile-de-France"
rename_columns$REGION[rename_columns$DEP %in% c("14","27","50","61","76")]                                    <- "Normandie"
rename_columns$REGION[rename_columns$DEP %in% c("16","17","19","23","24","33","40","47","64","79","86","87")] <- "Nouvelle-Aquitaine"
rename_columns$REGION[rename_columns$DEP %in% c("09","11","12","30","31","32","34","46","48","65","66","81","82")] <- "Occitanie"
rename_columns$REGION[rename_columns$DEP %in% c("44","49","53","72","85")]                                    <- "Pays de la Loire"
rename_columns$REGION[rename_columns$DEP %in% c("04","05","06","13","83","84")]                               <- "PACA"
rename_columns$REGION[rename_columns$DEP %in% c("972","974")]                                                 <- "DOM"
rename_columns$REGION <- as.factor(rename_columns$REGION)
table(rename_columns$REGION, useNA = "ifany")


   Auvergne-Rhone-Alpes Bourgogne-Franche-Comte                Bretagne 
                   1167                     329                     415 
    Centre-Val de Loire                   Corse                     DOM 
                    375                      43                     362 
              Grand Est         Hauts-de-France           Ile-de-France 
                    802                    1039                    3850 
              Normandie      Nouvelle-Aquitaine               Occitanie 
                    481                     753                     816 
                   PACA        Pays de la Loire 
                   1364                     598 

## Tableau par département
Les données sont au niveau du quartier. Pour comparer les départements, on calcule pour chacun la moyenne de ses IRIS. On obtient un tableau d'une ligne par département.

Pour D9/D1 et S80/S20, on prend la médiane et non la moyenne. Ces rapports deviennent énormes dans les quartiers où le 1er décile est proche de 0 €, et la moyenne n'aurait plus de sens.

In [15]:
# Create a new data frame 'd' with selected columns
vars <- c("MED","Q1","Q3","D1","D2","D3","D4","D6","D7","D8","D9",
          "TP60","PIMP","GINI","EQ","PACT","PTSA","PCHO","PBEN","PPEN","PAUT")

# Aggregate the data by department and region, calculating the mean for selected variables
dep <- aggregate(rename_columns[, vars], by = list(DEP = rename_columns$DEP, LIBDEP = rename_columns$LIBDEP, REGION = rename_columns$REGION), FUN = mean, na.rm = TRUE)

# Calculate the median of RD and S80S20 by department
dep_ratio <- aggregate(rename_columns[, c("RD", "S80S20")], by = list(DEP = rename_columns$DEP), FUN = median, na.rm = TRUE)

# Merge the median ratios back into the department-level data frame
dep <- merge(dep, dep_ratio, by = "DEP")

# Number of IRIS per department
dep$n_iris <- as.vector(table(rename_columns$DEP)[dep$DEP])
rownames(dep) <- dep$LIBDEP
dim(dep)
head(dep)

# Metropolitan France only (used for maps and models)
dep_m <- dep[dep$REGION != "DOM", ]
dep_m$REGION <- droplevels(dep_m$REGION)
dim(dep_m)

[1] 98 27

,DEP,LIBDEP,REGION,MED,Q1,Q3,D1,D2,D3,D4,⋯,EQ,PACT,PTSA,PCHO,PBEN,PPEN,PAUT,RD,S80S20,n_iris
,<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
Ain,01,Ain,Auvergne-Rhone-Alpes,19777.07,12268.293,28694.15,6202.683,10578.049,13850.00,16902.68,⋯,0.8390244,67.26585,61.00250,3.682500,2.685000,28.17317,4.560976,6.7,8.6,41
Aisne,02,Aisne,Hauts-de-France,16253.73,9220.448,24029.70,3440.597,7523.881,10758.36,13586.27,⋯,0.9761194,62.67910,55.11356,3.838983,4.494915,33.60597,3.714925,10.2,12.8,67
Allier,03,Allier,Auvergne-Rhone-Alpes,18615.11,11429.333,26237.33,4769.111,9611.778,13053.78,15906.67,⋯,0.8228889,55.59333,48.34444,3.008889,4.240000,40.34667,4.060000,7.8,9.8,45
Alpes-de-Haute-Provence,04,Alpes-de-Haute-Provence,PACA,19421.76,12282.941,27827.06,6168.824,10535.882,13817.06,16571.18,⋯,0.8429412,55.14706,47.61176,3.158824,4.376471,34.15294,10.700000,5.6,8.2,17
Hautes-Alpes,05,Hautes-Alpes,PACA,20479.05,13727.619,28201.90,8103.333,12083.810,15210.48,17931.90,⋯,0.7176190,59.69524,51.20952,3.080952,5.404762,34.28571,6.019048,4.4,6.2,21
Alpes-Maritimes,06,Alpes-Maritimes,PACA,21873.32,14001.677,31917.75,7629.880,12193.713,15663.77,18783.02,⋯,0.8350000,64.15299,55.39880,3.391265,5.321988,28.35329,7.493713,5.9,7.9,334


[1] 96 27

## Description, résumé statistique et visualisation

In [17]:
summary(rename_columns[, c("MED","TP60","GINI","RD","PTSA","PPEN")])

      MED             TP60            GINI              RD        
 Min.   : 2280   Min.   : 5.00   Min.   :0.1830   Min.   :   2.3  
 1st Qu.:16402   1st Qu.:16.70   1st Qu.:0.3280   1st Qu.:   4.8  
 Median :20860   Median :25.15   Median :0.3660   Median :   6.4  
 Mean   :21746   Mean   :29.27   Mean   :0.3735   Mean   :  18.5  
 3rd Qu.:25480   3rd Qu.:38.60   3rd Qu.:0.4110   3rd Qu.:   9.5  
 Max.   :73040   Max.   :90.00   Max.   :0.8050   Max.   :5140.2  
                 NA's   :1796                     NA's   :654     
      PTSA            PPEN      
 Min.   :17.90   Min.   :-0.40  
 1st Qu.:55.00   1st Qu.:19.40  
 Median :61.80   Median :25.20  
 Mean   :61.29   Mean   :25.92  
 3rd Qu.:68.30   3rd Qu.:31.80  
 Max.   :96.50   Max.   :65.90  
 NA's   :440                    

In [19]:
# Same table as in visualization_state.r, exported to Excel for the report
tem <- rename_columns[, c("MED","TP60","PIMP","D1","D9","RD","S80S20","GINI","PTSA","PCHO","PBEN","PPEN","PAUT")]

# Summary statistics
summary_stats <- data.frame(
  Min. = round(apply(tem, 2, min, na.rm = TRUE), 3),
  `1st Qu.` = round(apply(tem, 2, quantile, probs = 0.25, na.rm = TRUE), 3),
  Median = round(apply(tem, 2, median, na.rm = TRUE), 3),
  Mean = round(apply(tem, 2, mean, na.rm = TRUE), 3),
  `3rd Qu.` = round(apply(tem, 2, quantile, probs = 0.75, na.rm = TRUE), 3),
  Max. = round(apply(tem, 2, max, na.rm = TRUE), 3)
)
summary_stats

# Transpose the data frame
summary_stats <- t(summary_stats)

# Convert the transposed summary statistics to a data frame
summary_stats_df <- as.data.frame(summary_stats)

# Add a new column 'Measure' to the summary statistics data frame
summary_stats_df$Measure <- rownames(summary_stats_df)

# Reorder the columns to have 'Measure' as the first column
rownames(summary_stats_df) <- NULL
write_xlsx(summary_stats_df, path = "summary_stats.xlsx")

,Min.,X1st.Qu.,Median,Mean,X3rd.Qu.,Max.
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
MED,2280.000,16402.500,20860.000,21745.846,25480.000,73040.000
TP60,5.000,16.700,25.150,29.268,38.600,90.000
PIMP,8.600,40.300,52.100,51.702,63.700,89.100
D1,0.000,3530.000,6640.000,7029.300,10160.000,28440.000
D9,10990.000,32592.500,39810.000,45406.953,50850.000,392130.000
RD,2.300,4.800,6.400,18.500,9.500,5140.200
S80S20,2.500,6.300,8.600,1061.296,12.700,3324697.600
GINI,0.183,0.328,0.366,0.373,0.411,0.805
PTSA,17.900,55.000,61.800,61.286,68.300,96.500
